# Reproduce exact MusicFM NaN window
Mac, kernel SongFormer Python3.11, root Section-Annotations.
Hanya memeriksa window dari last_encoder_failure.json: input sama pada beberapa
panggilan, instance baru, serta transisi window panjang sebelumnya. Setelah itu
MusicFM PyTorch asli dijalankan pada audio yang sama dengan CPU FP32.
Audio tidak di-clip/dinormalisasi; cache training/model asli tidak berubah.
Array diagnostik disimpan lokal untuk reproduksi; cukup bagikan diagnostic.json.


In [ ]:
import subprocess, sys, json
from pathlib import Path
from datetime import datetime
assert Path('finetuning/config.json').is_file()
failure = Path('outputs.local/coreml-encoder-cache/last_encoder_failure.json')
r = json.loads(failure.read_text())
print(r['track_id'], r['start_sample']/r['sample_rate'], '->', r['end_sample']/r['sample_rate'])
output = Path('outputs.local') / ('musicfm-diagnostic-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
log_path = Path(str(output)+'.log')
command = [sys.executable, '-u', '-m', 'deployment.diagnose_musicfm',
    '--songmap-root', '/Users/danendra/Documents/ADA26/C6/SongMap',
    '--failure', str(failure), '--output-dir', str(output), '--repeats', '3']
with log_path.open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line)
            log.flush()
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
if code:
    raise RuntimeError(f'Proses gagal ({code}); lihat {log_path}. Laporan parsial mungkin ada di {output}.')


In [ ]:
report = json.loads((output/'diagnostic.json').read_text())
print('Core ML declared types:',report['coreml_precision'])
for name in ('same_instance','fresh_instance','after_preceding_window'):
    for item in report.get(name,[]):
        print(name, item['call'], '| NaN:',item['nan_count'],'| Inf:',item['inf_count'],
              '| bad frames:',item['bad_frame_count'], '| first indices:',item['bad_frame_indices'][:30])
print('Same-instance outputs identical:',report['same_instance_outputs_identical'])
print('Fresh instance bad mask matches:',report['fresh_instance_bad_mask_matches'])
if 'pytorch' in report:
    print('PyTorch NaN:',report['pytorch']['nan_count'],'| Inf:',report['pytorch']['inf_count'])
print('Share:',output/'diagnostic.json')
